# Kaggle: `dpo_from_base` -- DPO without SFT warm-start -> evaluate

Ablation run: same DPO training as `dpo_from_sft`, but starting from a **fresh LoRA on the base model** rather than warm-starting from the `sft_qlora` adapter (`--adapter` simply omitted from `train_dpo.py`'s args -- matches `config.yaml`'s `runs: dpo_from_base: warm_start: null`). Isolates DPO's own contribution: does preference optimization alone (without SFT first) get you most of the way there, or does it need the SFT warm-start to have something coherent to refine?

`baseline`, `sft_qlora`, and `dpo_from_sft` are already done (`results/summary.csv`) -- this notebook only adds the `dpo_from_base` row.

No adapter needs to be uploaded this time (no warm-start) -- simpler input set than `kaggle_dpo.ipynb`.

**Before running:**
1. Zip `src/` (contents) + `config.yaml` as `src.zip`.
2. Upload that zip plus `data/splits/dpo_train.jsonl`, `data/splits/dpo_valid.jsonl`, `data/splits/sft_test.jsonl` as a Kaggle Dataset.
3. Single T4 accelerator, Internet on.
4. Run all cells. Check the dry run's timing before letting the full run proceed -- same discipline as every prior notebook.

**Output:** `/kaggle/working/adapters/dpo_from_base/` (download to `adapters/dpo_from_base/` locally), `/kaggle/working/results/` (`dpo_from_base_gen.jsonl`, `dpo_from_base_scored.jsonl`, `summary.csv` with just the `dpo_from_base` row -- merge into local `results/summary.csv`).

In [ ]:
!pip install -q -U "trl==1.10.0" "peft==0.20.0" "transformers==5.15.0" "bitsandbytes==0.50.1" "accelerate==1.14.0" pyyaml
# peft 0.20.0 requires torchao>=0.16.0 for an internal LoRA-dispatch check;
# Kaggle's base image ships torchao==0.10.0. Only bites when loading a PEFT
# adapter onto a full-precision (non-4-bit) base model -- not the case in
# this notebook's own eval calls, but uninstalled defensively anyway since
# this project doesn't use torchao at all and it cost a full 3h15m training
# run + eval crash to find in kaggle_sft_lora_fp.ipynb. See LOG.md 2026-08-19.
!pip uninstall -y -q torchao
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Unsloth (accelerated QLoRA path)

Same install + CUDA-survived check as every other notebook. See `LOG.md` 2026-08-18/19: `--no-deps` is required (broke CUDA once without it), and it has *also* broken once even with `--no-deps` for an unrelated, transient Kaggle-side reason (likely GPU quota) -- if this assertion fails, check whether the *first* cell's CUDA line was already `False` before concluding it's the Unsloth install's fault.

In [ ]:
!pip install -q --no-deps unsloth unsloth_zoo
import torch
print("CUDA available after unsloth install:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), (
    "CUDA broke after installing unsloth -- see LOG.md 2026-08-18/19. Do not proceed with "
    "--use-unsloth training if this assertion fails; fall back to USE_UNSLOTH=False instead, or "
    "check whether the *first* cell's CUDA line was already False (a different, Kaggle-side issue)."
)

In [ ]:
import os, sys, zipfile

def find_repo(root="/kaggle/input"):
    repo_dir = None
    config_path = None
    zip_path = None
    for r, dirs, files in os.walk(root):
        if "build_irac.py" in files and os.path.basename(r) == "data":
            src_dir = os.path.dirname(r)
            if os.path.basename(src_dir) == "src":
                repo_dir = os.path.dirname(src_dir)
        if config_path is None and "config.yaml" in files:
            config_path = os.path.join(r, "config.yaml")
        if "src.zip" in files:
            zip_path = os.path.join(r, "src.zip")
    return repo_dir, config_path, zip_path

repo_dir, config_path, zip_path = find_repo()

if repo_dir is None and zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/kaggle/working/repo")
    repo_dir, config_path, _ = find_repo("/kaggle/working/repo")
    print("Extracted", zip_path)

if repo_dir is None or config_path is None:
    raise FileNotFoundError(
        f"repo_dir={repo_dir}, config_path={config_path} -- could not find both "
        "src/data/build_irac.py and config.yaml under /kaggle/input (in any "
        "nesting). Check the dataset is attached. If just attached/updated, "
        "try Restart & Run All."
    )

REPO_DIR = repo_dir
sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print("config.yaml at", config_path)

import yaml
with open(config_path) as f:
    cfg = yaml.safe_load(f)

MODEL_ID = cfg["model"]["candidates"][cfg["model"]["active"]]["hf_id"]
print("Active model:", MODEL_ID)

In [ ]:
import os

def find_data_file(name, root="/kaggle/input"):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(f"{name} not found under {root} -- check it was included in the uploaded dataset.")

DPO_TRAIN_FILE = find_data_file("dpo_train.jsonl")
DPO_VALID_FILE = find_data_file("dpo_valid.jsonl")
TEST_FILE = find_data_file("sft_test.jsonl")
print(DPO_TRAIN_FILE, DPO_VALID_FILE, TEST_FILE, sep="\n")

RESULTS_DIR = "/kaggle/working/results"
ADAPTERS_DIR = "/kaggle/working/adapters"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)
SUMMARY_CSV = os.path.join(RESULTS_DIR, "summary.csv")

from src.eval.generate import run as generate_run
from src.eval.score import score_file, append_summary_row

GEN_BATCH_SIZE = 16

## DPO training (`dpo_from_base`, no warm-start)

Hyperparameters from `config.yaml`'s `peft`/`training.dpo` sections, same as `dpo_from_sft` -- the *only* difference from that run is `--adapter` being omitted below, so `train_dpo.py` applies a fresh LoRA to the base model instead of continuing the SFT adapter's weights.

In [ ]:
peft_cfg = cfg["peft"]
dpo_cfg = cfg["training"]["dpo"]
max_seq_length = cfg["training"]["max_seq_length"]

DPO_FROM_BASE_DIR = os.path.join(ADAPTERS_DIR, "dpo_from_base")

USE_UNSLOTH = True

def train_dpo_args(output_dir, max_steps=None, num_epochs=None):
    args = [
        "--model", MODEL_ID,
        # NOTE: no --adapter here -- this is the entire point of this run.
        "--train-file", DPO_TRAIN_FILE,
        "--eval-file", DPO_VALID_FILE,
        "--output-dir", output_dir,
        "--load-in-4bit",
        "--lora-r", str(peft_cfg["lora_r"]),
        "--lora-alpha", str(peft_cfg["lora_alpha"]),
        "--lora-dropout", str(peft_cfg["lora_dropout"]),
        "--target-modules", *peft_cfg["target_modules"],
        "--max-seq-length", str(max_seq_length),
        "--beta", str(dpo_cfg["beta"]),
        "--per-device-batch-size", str(dpo_cfg["per_device_batch_size"]),
        "--gradient-accumulation-steps", str(dpo_cfg["gradient_accumulation_steps"]),
        "--learning-rate", str(dpo_cfg["learning_rate"]),
    ]
    if USE_UNSLOTH:
        args += ["--use-unsloth"]
    if max_steps is not None:
        args += ["--max-steps", str(max_steps)]
    else:
        args += ["--num-epochs", str(num_epochs)]
    return args

print(train_dpo_args(DPO_FROM_BASE_DIR, num_epochs=dpo_cfg["epochs"]))

In [ ]:
import subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["PYTHONUNBUFFERED"] = "1"

dryrun_dir = os.path.join(ADAPTERS_DIR, "dpo_from_base_dryrun")
cmd = [sys.executable, "-m", "src.train.train_dpo"] + train_dpo_args(dryrun_dir, max_steps=5)
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
# Full DPO training run. Check the dry-run cell's per-step timing above --
# the dpo_from_sft run landed around ~12.7s/step with this same stack
# (single-GPU pin + Unsloth + capability-aware bf16); this should land in
# the same ballpark since it's the same model/data/hyperparameters, just a
# different starting point for the LoRA weights. See LOG.md 2026-08-19.
cmd = [sys.executable, "-m", "src.train.train_dpo"] + train_dpo_args(DPO_FROM_BASE_DIR, num_epochs=dpo_cfg["epochs"])
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

## `dpo_from_base` eval

Same test split, same generation script, `dpo_from_base` adapter attached.

In [ ]:
DPO_FROM_BASE_GEN = os.path.join(RESULTS_DIR, "dpo_from_base_gen.jsonl")

generate_run(
    input_path=TEST_FILE,
    output_path=DPO_FROM_BASE_GEN,
    model_id=MODEL_ID,
    adapter_path=DPO_FROM_BASE_DIR,
    load_in_4bit=True,
    batch_size=GEN_BATCH_SIZE,
    max_new_tokens=350,
    limit=None,
)

In [ ]:
DPO_FROM_BASE_SCORED = os.path.join(RESULTS_DIR, "dpo_from_base_scored.jsonl")
dpo_from_base_summary = score_file(DPO_FROM_BASE_GEN, DPO_FROM_BASE_SCORED)
append_summary_row("dpo_from_base", dpo_from_base_summary, SUMMARY_CSV)
print("dpo_from_base:", dpo_from_base_summary)

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print(df.to_string(index=False))